In [8]:
import pandas as pd
from pathlib import Path
from collections import defaultdict
from IPython.display import display

In [9]:
folder = Path("/Users/kevinlin/Applied-Data-Science-Projekt/Data_project_app/downloads/wvv-pjs-2026/full_api_data")

JUMP_THRESHOLD_MIN = 5_000.0  # Mindest-Schwelle für große Distanzsprünge (Meter)
JUMP_FACTOR        = 5.0      # Multiplikator auf den Median


In [10]:
def load_parquet(path: Path, **kwargs) -> pd.DataFrame:
    """Liest eine Parquet-Datei; gibt leeres DataFrame zurück bei Fehler."""
    try:
        return pd.read_parquet(path, **kwargs)
    except Exception as e:
        print(f"Fehler beim Laden von {path.name}: {e}")
        return pd.DataFrame()
 
files = sorted(folder.glob("*.parquet"))
print(f"{len(files)} Dateien gefunden")


201 Dateien gefunden


Checking for empty columns

In [11]:
results = [
    {
        "datei":           file.name,
        "zeilen":          len(df := load_parquet(file)),
        "spalten":         len(df.columns),
        "fehlende_werte":  df.isna().sum().sum(),
        "duplikate":       df.duplicated().sum(),
    }
    for file in files
]
overview = pd.DataFrame(results)
 
print("Leere Dateien:")
for name in overview.loc[overview["zeilen"] == 0, "datei"]:
    print(f"  {name}")
 
display(overview)


Leere Dateien:
  data_2022-01-01_2022-12-31_line_23.parquet
  data_2022-01-01_2022-12-31_line_24.parquet
  data_2022-01-01_2022-12-31_line_41.parquet
  data_2023-01-01_2023-12-31_line_23.parquet
  data_2023-01-01_2023-12-31_line_24.parquet
  data_2023-01-01_2023-12-31_line_41.parquet
  data_2024-01-01_2024-12-31_line_23.parquet
  data_2024-01-01_2024-12-31_line_25.parquet
  data_2024-01-01_2024-12-31_line_35.parquet
  data_2024-01-01_2024-12-31_line_41.parquet
  data_2024-01-01_2024-12-31_line_7.parquet
  data_2024-01-01_2024-12-31_line_9.parquet
  data_2024-01-01_2024-12-31_line_99.parquet
  data_2025-01-01_2025-12-31_line_2.parquet
  data_2025-01-01_2025-12-31_line_23.parquet
  data_2025-01-01_2025-12-31_line_35.parquet
  data_2025-01-01_2025-12-31_line_41.parquet
  data_2025-01-01_2025-12-31_line_7.parquet
  data_2025-01-01_2025-12-31_line_9.parquet
  data_2025-01-01_2025-12-31_line_99.parquet
  data_2026-01-01_2026-12-31_line_2.parquet
  data_2026-01-01_2026-12-31_line_23.parquet
 

,datei,zeilen,spalten,fehlende_werte,duplikate
0,data_2022-01-01_2022-06-30_line_34.parquet,271043,42,1897301.0,0
1,data_2022-01-01_2022-06-30_line_5.parquet,400843,42,2805901.0,0
2,data_2022-01-01_2022-12-31_line_1.parquet,219212,42,1534484.0,0
3,data_2022-01-01_2022-12-31_line_10.parquet,389375,42,2725625.0,0
4,data_2022-01-01_2022-12-31_line_12.parquet,265047,42,1855329.0,0
...,...,...,...,...,...
196,data_2026-01-01_2026-12-31_line_95.parquet,2225,42,6751.0,0
197,data_2026-01-01_2026-12-31_line_96.parquet,3579,42,10373.0,0
198,data_2026-01-01_2026-12-31_line_97.parquet,3289,42,9313.0,0
199,data_2026-01-01_2026-12-31_line_98.parquet,1653,42,5624.0,0


In [12]:
column_stats: dict[str, dict] = {}
 
for file in files:
    df = load_parquet(file)
    if df.empty:
        continue
 
    for col in df.columns:
        stats = column_stats.setdefault(col, {"missing": 0, "total": 0})
        series = df[col]
 
        missing = int(series.isna().sum())
 
        # Leere Strings nur bei Textspalten zählen
        if pd.api.types.is_string_dtype(series) or series.dtype == object:
            # astype("string") konvertiert NaN → pd.NA (zählt nicht als "")
            empty_strings = int(
                series.astype("string").str.strip().eq("").sum()
            )
        else:
            empty_strings = 0
 
        stats["missing"] += missing + empty_strings
        stats["total"]   += len(df)
 
missing_summary = (
    pd.DataFrame([
        {
            "Spalte":                col,
            "Anzahl fehlender Werte": s["missing"],
            "Prozentualer Anteil":   round(100 * s["missing"] / s["total"], 2)
                                     if s["total"] else 0.0,
        }
        for col, s in column_stats.items()
    ])
    .sort_values(["Anzahl fehlender Werte", "Spalte"], ascending=[False, True])
    .reset_index(drop=True)
)
 
display(missing_summary)


,Spalte,Anzahl fehlender Werte,Prozentualer Anteil
0,next_station,21230401,96.58
1,next_station_short,21230401,96.58
2,day_group,21186434,96.38
3,edge,21186434,96.38
4,is_holiday,21186434,96.38
5,time_shift,21186434,96.38
6,unique_journey,21186434,96.38
7,arrival_plan_door,0,0.00
8,arrival_plan_stop,0,0.00
9,cumsum_distance_plan,0,0.00


Checking for negativ value 

In [14]:
COLS_REQUESTED = {
    "stop_event_id", "report_date", "line", "unique_journey",
    "stop_sequence", "station",
    "passenger_boarding", "passenger_exiting",
    "occupancy_departure", "vehicle_utilization", "quality_factor",
}
 
RULE_COLS = {
    "passenger_boarding < 0":   "passenger_boarding",
    "passenger_exiting < 0":    "passenger_exiting",
    "occupancy_departure < 0":  "occupancy_departure",
    "vehicle_utilization < 0":  "vehicle_utilization",
}

 
total_rows     = 0
summary_counts = {rule: 0 for rule in RULE_COLS}
violations     = []
quality_frames = []   # ← für die Quality-Factor-Analyse weiter unten
 
for p in files:
    df = load_parquet(p, columns=COLS_REQUESTED)
    if df.empty:
        continue
 
    df = df.reindex(columns=COLS_REQUESTED)
    total_rows += len(df)
 

    # Numerische Spalten sicherstellen
    num = {
        col: pd.to_numeric(df[col], errors="coerce")
        for col in RULE_COLS.values()
    }
 
    masks = {rule: num[col] < 0 for rule, col in RULE_COLS.items()}
 
    for rule, mask in masks.items():
        summary_counts[rule] += int(mask.sum())
 
    combined = pd.concat(masks.values(), axis=1).any(axis=1)
    if combined.any():
        violations.append(df.loc[combined].copy())
 
    # Quality-Factor-Daten sammeln (für Abschnitt 3b)
    if "quality_factor" in df.columns and df["quality_factor"].notna().any():
        quality_frames.append(df["quality_factor"])
 
summary_df = pd.DataFrame([
    {
        "rule":           rule,
        "violations":     count,
        "percent_total":  round(100.0 * count / total_rows, 4) if total_rows else 0.0,
    }
    for rule, count in summary_counts.items()
])
 
violations_df = (
    pd.concat(violations, ignore_index=True)
    if violations
    else pd.DataFrame(columns=COLS_REQUESTED)
)
 
violations_df.to_csv("dq_negative_values_violations.csv", index=False)
 
print("Zusammenfassung negativer Werte:")
display(summary_df)
print(f"\nGesamtzeilen: {total_rows:,}  |  Verletzte Zeilen: {len(violations_df):,}")
 


Fehler beim Laden von data_2022-01-01_2022-12-31_line_23.parquet: No match for FieldRef.Name(passenger_exiting) in __fragment_index: int32
__batch_index: int32
__last_in_fragment: bool
__filename: string
Fehler beim Laden von data_2022-01-01_2022-12-31_line_24.parquet: No match for FieldRef.Name(passenger_exiting) in __fragment_index: int32
__batch_index: int32
__last_in_fragment: bool
__filename: string
Fehler beim Laden von data_2022-01-01_2022-12-31_line_41.parquet: No match for FieldRef.Name(passenger_exiting) in __fragment_index: int32
__batch_index: int32
__last_in_fragment: bool
__filename: string
Fehler beim Laden von data_2023-01-01_2023-12-31_line_23.parquet: No match for FieldRef.Name(passenger_exiting) in __fragment_index: int32
__batch_index: int32
__last_in_fragment: bool
__filename: string
Fehler beim Laden von data_2023-01-01_2023-12-31_line_24.parquet: No match for FieldRef.Name(passenger_exiting) in __fragment_index: int32
__batch_index: int32
__last_in_fragment: bool

,rule,violations,percent_total
0,passenger_boarding < 0,64,0.0003
1,passenger_exiting < 0,69,0.0003
2,occupancy_departure < 0,1,0.0000
3,vehicle_utilization < 0,1,0.0000



Gesamtzeilen: 21,982,494  |  Verletzte Zeilen: 72


In [15]:
if quality_frames:
    qf_all = pd.concat(quality_frames, ignore_index=True)
 
    print("=== quality_factor (gesamt) ===")
    print(qf_all.describe().to_string())
    print()
    print("Häufigkeitsverteilung (Top 50):")
    print(qf_all.value_counts().sort_index().head(50).to_string())
 
    # Zeilen mit quality_factor == 0
    print("\n--- quality_factor == 0: Deskriptivstatistik der Passierspalten ---")
    # Wir laden noch mal nur die nötigen Spalten (leichtgewichtig)
    zero_qf_frames = []
    for p in files:
        df = load_parquet(p, columns=["quality_factor",
                                       "passenger_boarding",
                                       "passenger_exiting",
                                       "occupancy_departure"])
        if df.empty:
            continue
        mask_zero = df["quality_factor"] == 0
        if mask_zero.any():
            zero_qf_frames.append(df.loc[mask_zero])
 
    if zero_qf_frames:
        zero_qf = pd.concat(zero_qf_frames, ignore_index=True)
        display(zero_qf[["passenger_boarding", "passenger_exiting",
                          "occupancy_departure"]].describe())
 
    print("\n--- quality_factor ∈ [150, 1000): Deskriptivstatistik ---")
    high_qf_frames = []
    for p in files:
        df = load_parquet(p, columns=["quality_factor",
                                       "passenger_boarding",
                                       "passenger_exiting",
                                       "occupancy_departure"])
        if df.empty:
            continue
        mask_high = df["quality_factor"].between(150, 999, inclusive="left")
        if mask_high.any():
            high_qf_frames.append(df.loc[mask_high])
 
    if high_qf_frames:
        high_qf = pd.concat(high_qf_frames, ignore_index=True)
        display(high_qf[["passenger_boarding", "passenger_exiting",
                          "occupancy_departure"]].describe())
else:
    print("Keine quality_factor-Daten vorhanden.")


=== quality_factor (gesamt) ===
count    2.198249e+07
mean     2.312324e+01
std      3.792815e+01
min      0.000000e+00
25%      4.000000e+00
50%      9.000000e+00
75%      2.300000e+01
max      2.000000e+02

Häufigkeitsverteilung (Top 50):
quality_factor
0     2242639
1      690334
2     1094655
3     1183310
4     1239194
5     1113185
6     1080489
7      923327
8      819963
9      694702
10     669158
11     685308
12     516664
13     588704
14     396502
15     479230
16     280086
17     331724
18     377910
19     233192
20     243606
21     212439
22     327850
23     120209
24     184288
25     176680
26     141797
27     129674
28      46467
29     307061
30      98062
31      87824
32     120984
33     152089
34      69515
35      89223
36      80797
37      56000
38      73257
39      29633
40     292677
41      24252
42      60691
43      66868
44      66452
45      43883
46      74445
47      35568
48      58934
49      35432

--- quality_factor == 0: Deskriptivstatisti

,passenger_boarding,passenger_exiting,occupancy_departure
count,2.242639e+06,2.242639e+06,2.242639e+06
mean,1.365638e+00,1.371186e+00,9.260679e+00
std,3.920717e+00,3.868941e+00,1.450057e+01
min,-1.600000e+01,-1.900000e+01,-3.000000e+00
25%,0.000000e+00,0.000000e+00,1.000000e+00
50%,0.000000e+00,0.000000e+00,4.000000e+00
75%,1.000000e+00,1.000000e+00,1.100000e+01
max,7.810000e+02,7.550000e+02,3.730000e+02



--- quality_factor ∈ [150, 1000): Deskriptivstatistik ---
Fehler beim Laden von data_2022-01-01_2022-12-31_line_23.parquet: No match for FieldRef.Name(quality_factor) in __fragment_index: int32
__batch_index: int32
__last_in_fragment: bool
__filename: string
Fehler beim Laden von data_2022-01-01_2022-12-31_line_24.parquet: No match for FieldRef.Name(quality_factor) in __fragment_index: int32
__batch_index: int32
__last_in_fragment: bool
__filename: string
Fehler beim Laden von data_2022-01-01_2022-12-31_line_41.parquet: No match for FieldRef.Name(quality_factor) in __fragment_index: int32
__batch_index: int32
__last_in_fragment: bool
__filename: string
Fehler beim Laden von data_2023-01-01_2023-12-31_line_23.parquet: No match for FieldRef.Name(quality_factor) in __fragment_index: int32
__batch_index: int32
__last_in_fragment: bool
__filename: string
Fehler beim Laden von data_2023-01-01_2023-12-31_line_24.parquet: No match for FieldRef.Name(quality_factor) in __fragment_index: int32
_

,passenger_boarding,passenger_exiting,occupancy_departure
count,636728.000000,636728.000000,636728.000000
mean,0.106820,0.107569,0.891797
std,1.841882,1.682547,3.334699
min,-74.000000,-75.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000
max,497.000000,495.000000,131.000000


In [16]:
sequence_issues = []
 
for file in files:
    df = load_parquet(file)
    if df.empty:
        continue
    needed = {"unique_journey", "stop_sequence"}
    if not needed.issubset(df.columns):
        continue
 
    df = df.dropna(subset=["unique_journey"])
    df = df.sort_values(["unique_journey", "stop_sequence"])
 
    # Diff innerhalb jeder Fahrt – NaN markiert den ersten Stop jeder Fahrt
    grp_diff = df.groupby("unique_journey", sort=False)["stop_sequence"].diff()
 
    # Verstoß: diff ≤ 0 (nicht-streng-monoton steigend)
    mask = grp_diff.notna() & (grp_diff <= 0)
 
    if mask.any():
        bad = df.loc[mask, ["unique_journey", "stop_sequence"]].copy()
        bad["stop_sequence_prev"] = df.loc[mask, "stop_sequence"] - grp_diff[mask]
        bad["file"]               = file.name
        bad["violation_type"]     = "nicht_monoton_steigend"
        sequence_issues.append(bad)
 
issues_seq = (
    pd.concat(sequence_issues, ignore_index=True)
    if sequence_issues
    else pd.DataFrame()
)
 
issue_count = len(issues_seq)
print(f"Verstöße (stop_sequence nicht monoton steigend): {issue_count}")
if not issues_seq.empty:
    print(f"Betroffene Fahrten: {issues_seq['unique_journey'].nunique()}")
    display(issues_seq.head(20))
else:
    print("Keine Verstöße – stop_sequence ist korrekt monoton steigend!")
 


Verstöße (stop_sequence nicht monoton steigend): 0
Keine Verstöße – stop_sequence ist korrekt monoton steigend!


In [17]:
distance_issues = []
 
for file in files:
    df = load_parquet(file)
    if df.empty or "cumsum_distance_plan" not in df.columns or "stop_sequence" not in df.columns:
        continue
 
    # Gruppen-Schlüssel ermitteln
    if "unique_journey" in df.columns and df["unique_journey"].notna().any():
        group_key = "unique_journey"
    elif "journey" in df.columns:
        group_key = "journey"
    else:
        continue
 
    df = df.dropna(subset=[group_key]).sort_values([group_key, "stop_sequence"])
    ds = df["cumsum_distance_plan"]
 
    # Fehlende absolute Werte
    missing_mask = ds.isna()
    if missing_mask.any():
        bad = df.loc[missing_mask, [group_key, "stop_sequence"]].copy()
        bad["cumsum_distance_plan"] = np.nan
        bad["previous_distance"]    = np.nan
        bad["delta"]                = np.nan
        bad["issue_type"]           = "fehlender_wert"
        bad["file"]                 = file.name
        if "stop_event_id" in df.columns:
            bad["stop_event_id"] = df.loc[missing_mask, "stop_event_id"]
        distance_issues.append(bad)
 
    # Negative absolute Werte
    neg_mask = ds < 0
    if neg_mask.any():
        prev = df.groupby(group_key, sort=False)["cumsum_distance_plan"].shift(1)
        bad = df.loc[neg_mask, [group_key, "stop_sequence"]].copy()
        bad["cumsum_distance_plan"] = ds[neg_mask]
        bad["previous_distance"]    = prev[neg_mask]
        bad["delta"]                = ds[neg_mask] - prev[neg_mask]
        bad["issue_type"]           = "negativer_wert"
        bad["file"]                 = file.name
        if "stop_event_id" in df.columns:
            bad["stop_event_id"] = df.loc[neg_mask, "stop_event_id"]
        distance_issues.append(bad)
 
    # Diff innerhalb jeder Fahrt
    diff = df.groupby(group_key, sort=False)["cumsum_distance_plan"].diff()
 
    # Rücksprünge (diff < 0) – nur bei nicht-fehlenden Werten
    back_mask = diff.notna() & (diff < 0)
    if back_mask.any():
        prev = df["cumsum_distance_plan"].shift(1)
        bad = df.loc[back_mask, [group_key, "stop_sequence"]].copy()
        bad["cumsum_distance_plan"] = ds[back_mask]
        bad["previous_distance"]    = prev[back_mask]
        bad["delta"]                = diff[back_mask]
        bad["issue_type"]           = "ruecksprung"
        bad["file"]                 = file.name
        if "stop_event_id" in df.columns:
            bad["stop_event_id"] = df.loc[back_mask, "stop_event_id"]
        distance_issues.append(bad)
 
    # Ungewöhnlich große Sprünge (diff > Schwelle)
    pos_diff = diff[diff > 0]
    if not pos_diff.empty:
        threshold = max(JUMP_THRESHOLD_MIN, pos_diff.median() * JUMP_FACTOR)
        jump_mask = diff > threshold
        if jump_mask.any():
            prev = df["cumsum_distance_plan"].shift(1)
            bad = df.loc[jump_mask, [group_key, "stop_sequence"]].copy()
            bad["cumsum_distance_plan"] = ds[jump_mask]
            bad["previous_distance"]    = prev[jump_mask]
            bad["delta"]                = diff[jump_mask]
            bad["issue_type"]           = "grosser_sprung"
            bad["file"]                 = file.name
            if "stop_event_id" in df.columns:
                bad["stop_event_id"] = df.loc[jump_mask, "stop_event_id"]
            distance_issues.append(bad)
 
issues_cdp = (
    pd.concat(distance_issues, ignore_index=True)
    if distance_issues
    else pd.DataFrame()
)
 
if not issues_cdp.empty:
    issues_cdp.to_csv("cumsum_distance_plan_issues.csv", index=False, sep=";")
    print(f"{len(issues_cdp):,} Auffälligkeiten → cumsum_distance_plan_issues.csv")
    display(
        issues_cdp.groupby("issue_type")
        .size()
        .rename("Anzahl")
        .reset_index()
    )
else:
    print("Keine Auffälligkeiten in cumsum_distance_plan gefunden.")
 


165,378 Auffälligkeiten → cumsum_distance_plan_issues.csv


,issue_type,Anzahl
0,grosser_sprung,464
1,ruecksprung,164914


In [18]:
TIME_PAIRS = [
    ("arrival_plan_stop",  "departure_plan_stop",  "issue_stop_arrival_gt_departure"),
    ("arrival_plan_door",  "departure_plan_door",  "issue_door_arrival_gt_departure"),
]
EXTRA_COLS = ["stop_event_id", "unique_journey", "stop_sequence"]
 
time_violations = []
 
for p in files:
    df = load_parquet(p)
    if df.empty:
        continue
 
    # Datetime-Konvertierung für alle relevanten Spalten
    all_time_cols = {c for pair in TIME_PAIRS for c in pair[:2]}
    for col in all_time_cols & set(df.columns):
        if not pd.api.types.is_datetime64_any_dtype(df[col]):
            df[col] = pd.to_datetime(df[col], errors="coerce")
 
    # Violation-Masken aufbauen
    flag_masks: dict[str, pd.Series] = {}
    for arr_col, dep_col, flag_name in TIME_PAIRS:
        if arr_col in df.columns and dep_col in df.columns:
            both_valid = df[arr_col].notna() & df[dep_col].notna()
            flag_masks[flag_name] = both_valid & (df[arr_col] > df[dep_col])
 
    if not flag_masks:
        continue
 
    combined = pd.concat(flag_masks.values(), axis=1).any(axis=1)
    if not combined.any():
        continue
 
    # Nur vorhandene Extra-Spalten + Zeitspalten auswählen
    keep_cols = [c for c in EXTRA_COLS if c in df.columns]
    for arr_col, dep_col, _ in TIME_PAIRS:
        for c in (arr_col, dep_col):
            if c in df.columns:
                keep_cols.append(c)
 
    sub = df.loc[combined, keep_cols].copy()
    sub["file"] = p.name
 
    # FIX: Flags korrekt aus dem gefilterten Index übertragen
    for flag_name, mask in flag_masks.items():
        sub[flag_name] = mask.loc[combined].values  # ← war vorher buggy
 
    time_violations.append(sub.reset_index().rename(columns={"index": "row_index"}))
 
violations_time = (
    pd.concat(time_violations, ignore_index=True)
    if time_violations
    else pd.DataFrame()
)
 
print(f"Zeitliche Konsistenz – Verstöße gesamt: {len(violations_time):,}")
if not violations_time.empty:
    violations_time.to_csv("time_consistency_violations.csv", index=False)
    display(violations_time.head(20))
else:
    print("Keine Verstöße – alle Ankunftszeiten liegen vor den Abfahrtszeiten!")
 



Zeitliche Konsistenz – Verstöße gesamt: 0
Keine Verstöße – alle Ankunftszeiten liegen vor den Abfahrtszeiten!


In [19]:
from collections import defaultdict
 
station_map: dict = defaultdict(lambda: {
    "station_names":       set(),
    "station_short_names": set(),
    "rows":                [],
})
 
for file in files:
    df = load_parquet(file)
    if df.empty:
        continue
 
    needed = {"station_number", "station", "station_short"}
    if not needed.issubset(df.columns):
        continue
 
    subset = (
        df[list(needed)]
        .drop_duplicates()
        .dropna(subset=["station_number"])
        .copy()
    )
    subset["station"]       = subset["station"].astype(str).str.strip()
    subset["station_short"] = subset["station_short"].astype(str).str.strip()
 
    for _, row in subset.iterrows():
        entry = station_map[row["station_number"]]
        entry["station_names"].add(row["station"])
        entry["station_short_names"].add(row["station_short"])
        entry["rows"].append({
            "file":             file.name,
            "station_number":   row["station_number"],
            "station":          row["station"],
            "station_short":    row["station_short"],
        })
 
conflict_summary = []
conflict_details = []
 
for station_number, info in station_map.items():
    if len(info["station_names"]) > 1 or len(info["station_short_names"]) > 1:
        conflict_summary.append({
            "station_number":        station_number,
            "station_names":         " | ".join(sorted(info["station_names"])),
            "station_short_names":   " | ".join(sorted(info["station_short_names"])),
            "station_name_count":    len(info["station_names"]),
            "station_short_count":   len(info["station_short_names"]),
            "file_count":            len({r["file"] for r in info["rows"]}),
        })
        conflict_details.extend(info["rows"])
 
station_conflicts = (
    pd.DataFrame(conflict_summary)
    .sort_values(["station_name_count", "station_short_count", "station_number"],
                 ascending=[False, False, True])
    .reset_index(drop=True)
    if conflict_summary
    else pd.DataFrame()
)
 
station_conflict_details = (
    pd.DataFrame(conflict_details)
    .sort_values(["station_number", "file", "station", "station_short"])
    .reset_index(drop=True)
    if conflict_details
    else pd.DataFrame()
)
 
print(f"station_number-Konflikte: {len(station_conflicts)}")
if not station_conflicts.empty:
    display(station_conflicts)
    print("\nDetailliste:")
    display(station_conflict_details)
else:
    print("Keine Konflikte – jede station_number ist konsistent zugeordnet.")



station_number-Konflikte: 55


,station_number,station_names,station_short_names,station_name_count,station_short_count,file_count
0,292,Kürnachtalhalle | Kürnachtalhalle_1 | Odenwald...,KÜHA | KÜHA_1 | ODEN | ODEN_1,4,4,26
1,206,Jahnstraße | Jahnstraße_1 | Jahnstraße_2,JAHN | JAHN_1 | JAHN_2,3,3,26
2,1012,ABH | Betriebshof Heuchelhof | Betriebshof Heu...,ABH | ABH_1,3,2,14
3,20,Albert-Günther-Weg | Albert-Günther-Weg_1,AGÜW | AGÜW_1,2,2,2
4,23,Äußeres Hubland | Äußeres Hubland_1,ÄUHU | ÄUHU_1,2,2,34
5,27,Alfred-Nobel-Straße | Alfred-Nobel-Straße_1,ALNO | ALNO_1,2,2,15
6,33,Am Hubland | Campusbrücke,AHUB | CABR,2,2,15
7,36,Am Sand | Am Sand_1,AMSA | AMSA_1,2,2,5
8,43,Athener Ring | Athener Ring_1,ATHR | ATHR_1,2,2,29
9,64,Berliner Ring | Sozialgericht,BERL | SOGE,2,2,57



Detailliste:


,file,station_number,station,station_short
0,data_2022-01-01_2022-12-31_line_55.parquet,10,Betriebshof BBH,BBH
1,data_2023-01-01_2023-12-31_line_44.parquet,10,Betriebshof BBH,BBH
2,data_2023-01-01_2023-12-31_line_55.parquet,10,Betriebshof BBH,BBH
3,data_2024-01-01_2024-12-31_line_44.parquet,10,Betriebshof BBH,BBH
4,data_2024-01-01_2024-12-31_line_55.parquet,10,Betriebshof BBH,BBH
...,...,...,...,...
1459,data_2025-01-01_2025-12-31_line_3.parquet,1012,ABH,ABH
1460,data_2025-01-01_2025-12-31_line_3.parquet,1012,Betriebshof Heuchelhof,ABH
1461,data_2025-07-01_2025-12-31_line_5.parquet,1012,ABH,ABH
1462,data_2026-01-01_2026-12-31_line_3.parquet,1012,ABH,ABH
